In [ ]:

from pathlib import Path
import requests

DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"

PATH.mkdir(parents=True, exist_ok=True)

URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
    content = requests.get(URL + FILENAME).content
    (PATH / FILENAME).open("wb").write(content)


In [ ]:

import pickle
import gzip

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
    ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding="latin-1")


In [ ]:

import math
import matplotlib.pyplot as plt

x_train[0]
plt.figure(figsize=(2, 2))

w = int(math.sqrt(len(x_train[0])))

plt.imshow(x_train[0].reshape(w, w), cmap='gray')

plt.show()
print(x_train.shape)


In [ ]:

import torch

x_train, y_train, x_valid, y_valid = map(
    torch.tensor, (x_train, y_train, x_valid, y_valid)
)
n, c = x_train.shape

print(x_train, y_train)
print(x_train.shape)
print(y_train.min(), y_train.max())


In [ ]:

import math

weights = torch.randn(784, 10) / math.sqrt(784)
weights.requires_grad_()
bias = torch.zeros(10, requires_grad=True)


In [ ]:

fig, axs = plt.subplots(2, 5, figsize=(7, 7))

for i in range(2):
    for j in range(5):
       axs[i, j].imshow(weights[:, i * 5 + j].reshape(28, 28).detach().numpy())




In [ ]:

def log_softmax(x):
    return x - x.exp().sum(-1).log().unsqueeze(-1)

def model(xb):
    return log_softmax(xb @ weights + bias)


In [ ]:

bs = 64

xb = x_train[:bs]

preds = model(xb)

print(preds[0], preds.shape)


In [ ]:

def nll(input, target):
    return -input[range(target.shape[0]), target].mean()

loss_fn = nll



In [ ]:

yb = y_train[:bs]
print(loss_fn(preds, yb))


In [ ]:

def accuracy(out, yb):
    preds = torch.argmax(out, dim=1)
    return (preds == yb).float().mean()


print(accuracy(preds, yb))



In [ ]:

from IPython.core.debugger import set_trace

lr = 0.5
epochs = 10

for epoch in range(epochs):
    for i in range((n-1) // bs + 1):
        # set_trace()
        start_i = i * bs
        end_i = start_i + bs
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]
        pred = model(xb)
        loss = loss_fn(pred, yb)

        loss.backward()

        with torch.no_grad():
            weights -= weights.grad * lr
            bias -= bias.grad * lr
            weights.grad.zero_() #
            weights.grad.zero_()


print(loss_fn(model(xb), yb), accuracy(model(xb), yb))



In [ ]:

fig, axs = plt.subplots(2, 5, figsize=(7, 7))

for i in range(2):
    for j in range(5):
       axs[i, j].imshow(weights[:, i * 5 + j].reshape(28, 28).detach().numpy())

#plt.show()


# Refactoring Code Using the torch.nn Library

In [ ]:

import torch.nn.functional as F

loss_fn = F.cross_entropy

def model(xb):
    return xb @ weights + bias

print(loss_fn(model(xb), yb), accuracy(model(xb), yb))


In [ ]:

from torch import nn

class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(784, 10) / math.sqrt(784), requires_grad=True)
        self.bias = nn.Parameter(torch.zeros(10), requires_grad=True)

    def forward(self, xb):
        return xb @ self.weights + self.bias


model = Mnist_Logistic()
print(loss_fn(model(xb), yb))


In [ ]:

epochs = 2

def fit():

    for epoch in range(epochs):
        for i in range((n-1) // bs + 1):
            start_i = i * bs
            end_i = start_i + bs

            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()

            with torch.no_grad():
                for p in model.parameters(): p -= p.grad * lr
                model.zero_grad()


fit()
print(loss_fn(model(xb), yb))


In [ ]:

class Mnist_Logistic_Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(in_features=784, out_features=10)

    def forward(self, xb):
        return self.lin(xb)

model = Mnist_Logistic_Linear()
print(loss_fn(model(xb), yb))


In [ ]:

fit()

print(loss_fn(model(xb), yb))



In [ ]:

from torch import optim

def get_model():
    model = Mnist_Logistic_Linear()
    return model, optim.SGD(model.parameters(), lr=lr)

model, opt = get_model()
print(loss_fn(model(xb), yb))


for epoch in range(epochs):
    for i in range((n-1) // bs + 1):
        start_i = i * bs
        end_i = start_i + bs

        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]

        pred = model(xb)
        loss = loss_fn(pred, yb)

        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_fn(model(xb), yb))



In [ ]:

from torch.utils.data import TensorDataset

train_ds = TensorDataset(x_train, y_train)


In [ ]:

import torch.nn.functional as F

model, opt = get_model()

loss_fn = F.cross_entropy

for epoch in range(epochs):
    for i in range((n-1) // bs + 1):
        xb, yb = train_ds[i * bs: i * bs + bs]
        pred = model(xb)
        loss = loss_fn(pred, yb)
        
        loss.backward()

        opt.step()
        opt.zero_grad()

print(loss_fn(model(xb), yb))



In [ ]:

from torch.utils.data import DataLoader

model, opt = get_model()

train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs)

epochs = 2

for epoch in range(epochs):
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()

print(loss_fn(model(xb), yb))


In [ ]:

train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)

valid_ds = TensorDataset(x_valid, y_valid)
valid_dl = DataLoader(valid_ds, batch_size=2*bs)


In [ ]:

model, opt = get_model()

for epoch in range(epochs):
    model.train()
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()

    model.eval()
    with torch.no_grad():
        valid_loss = sum(loss_fn(model(xb), yb) for xb, yb in valid_dl)

    print(epoch, valid_loss / len(valid_dl))


In [ ]:

import numpy as np

def loss_batch(model, loss_fn, xb, yb, opt=None):
    loss = loss_fn(model(xb), yb)
    
    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(xb)

def fit(epochs, model, loss_fn, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            loss_batch(model, loss_fn, xb, yb, opt)

        model.eval()
        with torch.no_grad():
            losses, nums = zip(
                *[loss_batch(model, loss_fn, xb, yb) for xb, yb in valid_dl]
            )

        val_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)

        print(epoch, val_loss)

def get_data(train_ds, valid_ds, bs):
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True),
        DataLoader(valid_ds, batch_size=2*bs)
    )


In [ ]:

train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
model, opt = get_model()
fit(epochs, model, loss_fn, opt, train_dl, valid_dl)



In [ ]:

class Mnist_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.c2 = nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1)
        self.c3 = nn.Conv2d(16, 10, kernel_size=3, stride=2, padding=1)

    def forward(self, xb):
        xb = xb.view(-1, 1, 28, 28)
        xb = F.relu(self.c1(xb))
        xb = F.relu(self.c2(xb))
        xb = F.relu(self.c3(xb))
        xb = F.avg_pool2d(xb, 4)
        return xb.view(-1, xb.size(1))

lr=0.1

model = Mnist_CNN()
opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

fit(epochs, model, loss_fn, opt, train_dl, valid_dl)
